# COMP723 Lab 12 – Relation Extraction: *who (from where / which organisation) said what*

**Goal:** use NLTK's relation-extraction tools (`ne_chunk` + `extract_rels`) on the terrorism news dataset to output
`<speaker name>, <organisation>, <sentence>` for sentences that use a form of **"say"**.

**How to run in Colab:** `Runtime → Run all`. When asked, upload `terrorism.zip` (the lab dataset).

Notebook layout:
1. Setup and data loading
2. The lecture demo, with notes on how `extract_rels` works
3. Information-extraction task (the lab)
4. Results, evaluation and limitations

## 1. Setup

In [1]:
import nltk, re, os, glob, zipfile, random
import pandas as pd

# NLTK >= 3.9 renamed several resources (punkt_tab, *_eng, *_tab); download both names so this works on any version.
for pkg in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words', 'treebank']:
    nltk.download(pkg, quiet=True)
print('NLTK', nltk.__version__)

NLTK 3.10.3


In [2]:
# Load the dataset: upload terrorism.zip once (or put it in the working folder / Google Drive and change ZIP_PATH).
ZIP_PATH = 'terrorism.zip'
DATA_DIR = 'terrorism'

if not os.path.isdir(DATA_DIR):
    if not os.path.exists(ZIP_PATH):
        from google.colab import files          # only available in Colab
        uploaded = files.upload()                # choose terrorism.zip
        ZIP_PATH = next(iter(uploaded))
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

FILES = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.txt'), recursive=True))
print(len(FILES), 'articles')

1780 articles


## 2. Lecture demo – how `extract_rels` works

`extract_rels(subj, obj, tree, corpus='ace', pattern=..., window=...)`:
1. takes a sentence that `ne_chunk` has already tagged with named entities (PERSON, ORGANIZATION, GPE…);
2. splits it into **pairs of neighbouring named entities** and the words between them (the *filler*);
3. keeps a pair only when the subject/object types match, the filler has at most `window` tokens,
   and the regex `pattern` matches the filler.

Note: the filler is matched **with its POS tags**, e.g. `'is/VBZ from/IN'`. Only *adjacent* entities are paired,
so the object must be the first named entity after the subject.

In [3]:
from nltk import ne_chunk, pos_tag, word_tokenize
from nltk.sem.relextract import extract_rels, rtuple

sentence = "Johnny is from Auckland, Tom is from Wellington but Mary is from Christchurch which is in South Island."
chunked = ne_chunk(pos_tag(word_tokenize(sentence)))
print(chunked)
ISFROM = re.compile(r'.*from*')
for rel in extract_rels('PER', 'GPE', chunked, corpus='ace', pattern=ISFROM):
    print(rtuple(rel))

(S
  (PERSON Johnny/NNP)
  is/VBZ
  from/IN
  (GPE Auckland/NNP)
  ,/,
  (PERSON Tom/NNP)
  is/VBZ
  from/IN
  (GPE Wellington/NNP)
  but/CC
  (PERSON Mary/NNP)
  is/VBZ
  from/IN
  (GPE Christchurch/NNP)
  which/WDT
  is/VBZ
  in/IN
  (GPE South/NNP Island/NNP)
  ./.)
[PER: 'Johnny/NNP'] 'is/VBZ from/IN' [GPE: 'Auckland/NNP']
[PER: 'Tom/NNP'] 'is/VBZ from/IN' [GPE: 'Wellington/NNP']
[PER: 'Mary/NNP'] 'is/VBZ from/IN' [GPE: 'Christchurch/NNP']


In [4]:
# Demo II – Penn Treebank sample with the ROLE pattern (PER, <role> of/at ORG)
ROLE = re.compile(r'.*(chairman|president|trader|scientist|economist|analyst|partner).*')
for i, sent in enumerate(nltk.corpus.treebank.tagged_sents()[:500]):
    sent = nltk.ne_chunk(sent)
    for rel in extract_rels('PER', 'ORG', sent, corpus='ace', pattern=ROLE, window=8):
        print('{0:<5}{1}'.format(i, rtuple(rel)))

1    [PER: 'Vinken/NNP'] 'is/VBZ chairman/NN of/IN' [ORG: 'Elsevier/NNP']


254  [PER: 'Shugart/NNP'] ',/, currently/RB chairman/NN of/IN' [ORG: 'Seagate/NNP Technology/NNP']


325  [PER: 'George/NNP Foot/NNP'] ',/, a/DT managing/VBG partner/NN at/IN' [ORG: 'Newgate/NNP Management/NNP Associates/NNP']


331  [PER: 'Michael/NNP Porter/NNP'] ',/, an/DT analyst/NN at/IN' [ORG: 'Smith/NNP Barney/NNP']


391  [PER: 'Elliott/NNP Platt/NNP'] ',/, an/DT economist/NN at/IN' [ORG: 'Donaldson/NNP']


485  [PER: 'Arafat/NNP'] 'has/VBZ written/VBN to/TO the/DT chairman/NN of/IN the/DT' [ORG: 'International/NNP']


## 3. Information-extraction task: who from where / which organisation said what

The demo uses two kinds of pattern, and I use one of each:

| Pattern | Idea (from the demo) | Matches e.g. |
|---|---|---|
| **SAID** | like `ISFROM`: the relation word is in the filler | `Mr. Carter` **said** … `Pentagon` |
| **AFFIL** | like `ROLE`: `PER, <title> of/for/at ORG`, used only on sentences containing a form of *say* | `Naeem Baig`**, the president of the** `Islamic Circle` … said |

Design choices:
- **Forms of "said"** = `say / says / said / saying`, the forms the brief asks for. `told` and `stated` can be added to
  `SAY_WORDS` with a one-line change. They are left out because `told` usually has the
  listener as its object ("told **reporters**"), which adds noise.
- The pattern requires a **verb POS tag** (`said/VBD`, `says/VBZ`…), so the noun ("have **a say**") is not counted.
- **Organisation column:** the brief asks "from *where* **or** *which organisation*", but the output has one slot.
  The slot gets the ORGANIZATION. If there is none, it gets the GPE (place), and `entity_type` records which one was used.
  Using ORG only would drop the "where" half of the question.
- If a speaker matches several times in one sentence, keep one row: **AFFIL over SAID** (AFFIL is more precise, see Section 4) and **ORG over GPE**.

In [5]:
from nltk.tag import PerceptronTagger
from nltk.sem.relextract import extract_rels, rtuple

# Speed: in recent NLTK, nltk.ne_chunk() reloads the chunker model on every call (~0.5 s per sentence).
# Load the tagger and chunker once instead (~100x faster), with a fallback for older NLTK versions.
tagger = PerceptronTagger()
try:
    from nltk.chunk import ne_chunker
    _chunker = ne_chunker()
    chunk = _chunker.parse
except ImportError:
    chunk = nltk.ne_chunk

SAY_WORDS = ['say', 'says', 'said', 'saying']          # add 'told', 'stated' here to widen the net
SAY = '(' + '|'.join(SAY_WORDS) + ')'

SAY_SENT = re.compile(r'\b' + SAY + r'\b', re.I)       # cheap pre-filter on raw sentence text
SAY_VERB = re.compile(r'^' + SAY + r'$', re.I)           # token check, combined with a VB* tag
SAID  = re.compile(r'.*\b' + SAY + r'/VB[DGPZ]?\b.*', re.I)
# AFFIL: an appositive ", a spokesman for the" / ", the director of", or a bare "of" ("Senator X of Kentucky")
AFFIL = re.compile(r'^(,/, ([^ ]+/(DT|JJ|JJS|NN|NNS|NNP|VBG|CD|RB|POS|CC) )*'
                   r'(of|for|at|from|with|in)/IN|of/IN)( the/DT)?$', re.I)

def untag(s):
    """'Mr./NNP Carter/NNP' -> 'Mr. Carter'"""
    return ' '.join(w.rsplit('/', 1)[0] for w in s.split())

def clean(text):
    """Fix escaped quotes and drop the metadata lines (Word count / Copyright / Credit / CORRECTION)."""
    text = text.replace('\\"', '"').strip().strip('"')
    return '\n'.join(l for l in text.splitlines()
                     if not re.match(r'\s*(Word count:|Copyright|Credit:|CORRECTION)', l))

def extract_said(sentence):
    tagged = tagger.tag(word_tokenize(sentence))
    if not any(SAY_VERB.match(w) and t.startswith('VB') for w, t in tagged):
        return []
    tree = chunk(tagged)
    rows = []
    for name, pattern, window in [('AFFIL', AFFIL, 8), ('SAID', SAID, 10)]:
        for obj in ['ORG', 'GPE']:
            for r in extract_rels('PER', obj, tree, corpus='ace', pattern=pattern, window=window):
                rows.append({'speaker': untag(r['subjtext']), 'organisation': untag(r['objtext']),
                             'entity_type': obj, 'pattern': name, 'sentence': ' '.join(sentence.split()),
                             'relation': rtuple(r)})
    return rows

print(extract_said('"People are losing their sleep," said Naeem Baig, the president of the Islamic Circle of North America.'))

[{'speaker': 'Naeem Baig', 'organisation': 'Islamic Circle', 'entity_type': 'ORG', 'pattern': 'AFFIL', 'sentence': '"People are losing their sleep," said Naeem Baig, the president of the Islamic Circle of North America.', 'relation': "[PER: 'Naeem/NNP Baig/NNP'] ',/, the/DT president/NN of/IN the/DT' [ORG: 'Islamic/NNP Circle/NNP']"}]


In [6]:
%%time
rows, n_sent, n_say = [], 0, 0
for f in FILES:
    with open(f, encoding='utf-8', errors='ignore') as fh:
        text = clean(fh.read())
    for s in nltk.sent_tokenize(text):
        n_sent += 1
        if SAY_SENT.search(s):
            n_say += 1
            for r in extract_said(s):
                r['file'] = os.path.basename(f)
                rows.append(r)

raw = pd.DataFrame(rows)
# one row per (sentence, speaker): AFFIL beats SAID, ORG beats GPE
raw['rank'] = (raw.pattern != 'AFFIL') * 2 + (raw.entity_type != 'ORG')
df = (raw.sort_values('rank').drop_duplicates(['file', 'sentence', 'speaker'])
         .sort_values(['pattern', 'file']).drop(columns='rank').reset_index(drop=True))
print(f'{n_sent} sentences, {n_say} contain a form of "say", {len(df)} relations extracted')
print(df.groupby(['pattern', 'entity_type']).size())

68090 sentences, 15579 contain a form of "say", 825 relations extracted
pattern  entity_type
AFFIL    GPE            208
         ORG            311
SAID     GPE            194
         ORG            112
dtype: int64
CPU times: user 1min 27s, sys: 60.7 ms, total: 1min 27s
Wall time: 1min 28s


## 4. Results in the required format `<speaker name>, <organisation>, <sentence>`

In [7]:
for _, r in df.head(30).iterrows():
    print(f'{r.speaker}, {r.organisation}, {r.sentence}\n')

Howard Shatz, RAND Corporation, "There's a lot less certainty about how much money actually evaporated," said Howard Shatz, a senior economist at the RAND Corporation who has studied the Islamic State's finances.

Jonathan Schanzer, Treasury, "Defeating the formal military presence of a terrorist group will not significantly mitigate the threat of lone wolf or small independent cells that are based in the West," said Jonathan Schanzer, a former terrorism finance analyst at the Treasury Department who is now with the Foundation for Defense of Democracies in Washington.

Yakubu Nkeki, Abducted Chibok Girls Parents, The parents of a handful of missing girls have in past months received calls from phone numbers that belonged to their daughters, said Yakubu Nkeki, the chairman of the Abducted Chibok Girls Parents' Movement for Rescue.

Laurent Ledoux, Federal Public Service, The tipping point may have come Thursday, when Laurent Ledoux, president of the Federal Public Service for Mobility a

In [8]:
# Save everything (the CSV keeps the extra columns: entity_type, pattern, file, raw relation tuple)
df.to_csv('said_relations.csv', index=False)
with open('said_relations.txt', 'w') as fh:
    for _, r in df.iterrows():
        fh.write(f'{r.speaker}, {r.organisation}, {r.sentence}\n')
print('wrote said_relations.csv and said_relations.txt')
# In Colab, uncomment to download:
# from google.colab import files; files.download('said_relations.csv')

wrote said_relations.csv and said_relations.txt


### Spot-check: how accurate is each pattern?
Below is a random sample from each pattern, to judge by hand whether *organisation* really is the speaker's affiliation.

In [9]:
for p in ['SAID', 'AFFIL']:
    print('=' * 30, p)
    for _, r in df[df.pattern == p].sample(20, random_state=42).iterrows():
        print(f'- {r.speaker} | {r.organisation} | {r.sentence[:160]}')

============================== SAID
- Mohammad | FBI | Mohammad said the FBI informant did not record Farah speaking of any allegiance to the terror group.
- Carter | Pentagon | Carter said the Pentagon planned to address the shortfall in the Pentagon's 2017 budget.
- John G. Koeltl | Israel | Judge John G. Koeltl, writing for the panel, said the attacks in Israel had not been "expressly aimed at the United States," and evidence presented by the plain
- Obama | Islamic | President Obama said Thursday that the Islamic State appears to be shifting its tactics to "high profile" attacks on tourists in cities around the world as it l
- Farid | Canadian | "He was helping deaf people," Mr. Amer-Ouali's son Farid said on Canadian television, describing his father as an avid mountain climber who had fallen in love w
- Moniquet | French | Mr. Moniquet said he had found statements on the Internet published in French the same day as the Brussels attacks by the Al Wafaa Foundation, a media organiza

### Evaluation and limitations

I hand-checked the 20-row sample per pattern printed above (`random_state=42`). A row counts as correct when *organisation* really is
the speaker's affiliation **and** that person is the one speaking in the sentence.

| Pattern | Correct | Main error |
|---|---|---|
| **SAID** (demo's literal `PER … said … ORG`) | **3 / 20** | The entity after "said" is the **topic of the statement**, not the speaker's organisation, e.g. *"Molins said the **Peugeot** contained fingerprints"*. The 3 correct rows (Carter→Pentagon, Pelosi→Democrats, Erdogan→Turkey) are right by coincidence. |
| **AFFIL** (demo's `ROLE`-style appositive) | **18 / 20** | *"the sister of **Vincent Smith**"* (a person labelled ORG). *Andrew Rhoades* is mentioned but not speaking. |

Other limitations:
- **Entity boundaries come from NLTK's NE chunker and are often too short** (about 5 of the 18 correct AFFIL rows): "George" for George Washington
  University, "University" for University of Michigan, "Homeland" for Homeland Security. Some names are mis-typed (PER/ORG/GPE).
  The pattern step can only be as good as the chunker.
- **Recall is low**: only 825 relations from about 15,600 sentences that contain a form of *say*. `extract_rels` only pairs *adjacent* entities, and most quotes
  use a pronoun ("he said") or a speaker with no stated organisation ("Mr. Meiloud said"). Linking these to an
  earlier sentence would need coreference resolution, which `extract_rels` does not do.
- 20 rows per pattern is a small sample. Treat 15% vs 90% as a clear difference in direction, not exact precision figures.

**Conclusion:** use the **AFFIL** rows to answer "who from which organisation/where said what". The **SAID** rows are kept (see the `pattern`
column) to show what the demo's literal pattern produces and why it is not enough on its own.